In [ ]:
import customtkinter as ctk
from tkinter import filedialog, messagebox
import cv2
import numpy as np
from PIL import Image


# ============================================================
# APPEARANCE
# ============================================================

ctk.set_appearance_mode("light")


class ImageProcessingApp(ctk.CTk):

    def __init__(self):
        super().__init__()

        # ====================================================
        # WINDOW SETTINGS
        # ====================================================

        self.title("Image Processing Studio")
        self.geometry("1450x900")
        self.minsize(1100, 700)

        # ====================================================
        # BRIGHT COLOR PALETTE
        # ====================================================

        self.WHITE = "#FFFFFF"
        self.LIGHT_BG = "#F8FAFC"

        self.CYAN = "#06B6D4"
        self.CYAN_HOVER = "#0891B2"
        self.CYAN_LIGHT = "#CFFAFE"

        self.MAGENTA = "#D946EF"
        self.MAGENTA_HOVER = "#C026D3"
        self.MAGENTA_LIGHT = "#FAE8FF"

        self.YELLOW = "#FACC15"
        self.YELLOW_HOVER = "#EAB308"
        self.YELLOW_LIGHT = "#FEF9C3"

        self.ORANGE = "#F97316"
        self.ORANGE_HOVER = "#EA580C"
        self.ORANGE_LIGHT = "#FFEDD5"

        self.PURPLE = "#8B5CF6"
        self.PURPLE_HOVER = "#7C3AED"
        self.PURPLE_LIGHT = "#EDE9FE"

        self.GREEN = "#22C55E"
        self.GREEN_HOVER = "#16A34A"
        self.GREEN_LIGHT = "#DCFCE7"

        self.PINK = "#F43F5E"
        self.PINK_HOVER = "#E11D48"
        self.PINK_LIGHT = "#FFE4E6"

        self.BLUE = "#3B82F6"
        self.BLUE_HOVER = "#2563EB"
        self.BLUE_LIGHT = "#DBEAFE"

        self.DARK_TEXT = "#172033"
        self.GRAY_TEXT = "#64748B"
        self.BORDER = "#CBD5E1"

        self.configure(
            fg_color=self.LIGHT_BG
        )

        # ====================================================
        # VARIABLES
        # ====================================================

        self.image = None
        self.gray_image = None
        self.output_image = None

        self.selected_operator = "Mean"

        # ====================================================
        # HEADER
        # ====================================================

        header = ctk.CTkFrame(
            self,
            fg_color=self.WHITE,
            corner_radius=0
        )

        header.pack(
            fill="x"
        )

        # Color strip
        color_strip = ctk.CTkFrame(
            header,
            height=8,
            fg_color=self.CYAN,
            corner_radius=0
        )

        color_strip.pack(
            fill="x"
        )

        title = ctk.CTkLabel(
            header,
            text="🖼  IMAGE PROCESSING STUDIO",
            font=("Arial", 30, "bold"),
            text_color=self.DARK_TEXT
        )

        title.pack(
            pady=(18, 3)
        )

        subtitle = ctk.CTkLabel(
            header,
            text="Filters  •  Edge Detection  •  Pixel Analysis",
            font=("Arial", 14),
            text_color=self.GRAY_TEXT
        )

        subtitle.pack(
            pady=(0, 18)
        )

        # ====================================================
        # TOOLBAR
        # ====================================================

        toolbar = ctk.CTkFrame(
            self,
            fg_color=self.WHITE,
            corner_radius=15,
            border_width=1,
            border_color=self.BORDER
        )

        toolbar.pack(
            fill="x",
            padx=25,
            pady=12
        )

        upload_button = ctk.CTkButton(
            toolbar,
            text="📁  UPLOAD IMAGE",
            width=180,
            height=45,
            corner_radius=12,
            fg_color=self.CYAN,
            hover_color=self.CYAN_HOVER,
            text_color=self.WHITE,
            font=("Arial", 13, "bold"),
            command=self.upload_image
        )

        upload_button.pack(
            side="left",
            padx=12,
            pady=10
        )

        reset_button = ctk.CTkButton(
            toolbar,
            text="🔄  RESET",
            width=130,
            height=45,
            corner_radius=12,
            fg_color=self.PINK,
            hover_color=self.PINK_HOVER,
            text_color=self.WHITE,
            font=("Arial", 13, "bold"),
            command=self.reset
        )

        reset_button.pack(
            side="left",
            padx=5
        )

        self.image_info = ctk.CTkLabel(
            toolbar,
            text="No image loaded",
            font=("Arial", 13, "bold"),
            text_color=self.GRAY_TEXT
        )

        self.image_info.pack(
            side="right",
            padx=20
        )

        # ====================================================
        # SCROLLABLE MAIN AREA
        # ====================================================

        main_area = ctk.CTkScrollableFrame(
            self,
            fg_color=self.LIGHT_BG,
            corner_radius=0
        )

        main_area.pack(
            fill="both",
            expand=True,
            padx=0,
            pady=0
        )

        # ====================================================
        # IMAGE CONTAINER
        # ====================================================

        image_container = ctk.CTkFrame(
            main_area,
            fg_color=self.LIGHT_BG,
            height=390
        )

        image_container.pack(
            fill="x",
            padx=25,
            pady=(5, 10)
        )

        image_container.pack_propagate(False)

        # ====================================================
        # INPUT IMAGE CARD
        # ====================================================

        input_card = ctk.CTkFrame(
            image_container,
            fg_color=self.WHITE,
            corner_radius=20,
            border_width=3,
            border_color=self.CYAN
        )

        input_card.pack(
            side="left",
            fill="both",
            expand=True,
            padx=(0, 8)
        )

        input_title = ctk.CTkLabel(
            input_card,
            text="📷  INPUT IMAGE",
            font=("Arial", 18, "bold"),
            text_color=self.CYAN
        )

        input_title.pack(
            pady=(12, 5)
        )

        self.input_image_label = ctk.CTkLabel(
            input_card,
            text="📷\n\nUpload an image to begin",
            font=("Arial", 17),
            text_color=self.GRAY_TEXT
        )

        self.input_image_label.pack(
            fill="both",
            expand=True,
            padx=15,
            pady=10
        )

        # ====================================================
        # OUTPUT IMAGE CARD
        # ====================================================

        output_card = ctk.CTkFrame(
            image_container,
            fg_color=self.WHITE,
            corner_radius=20,
            border_width=3,
            border_color=self.GREEN
        )

        output_card.pack(
            side="right",
            fill="both",
            expand=True,
            padx=(8, 0)
        )

        output_title = ctk.CTkLabel(
            output_card,
            text="✨  PROCESSED OUTPUT",
            font=("Arial", 18, "bold"),
            text_color=self.GREEN
        )

        output_title.pack(
            pady=(12, 5)
        )

        self.output_image_label = ctk.CTkLabel(
            output_card,
            text="⚙️\n\nChoose an operator\nand apply it",
            font=("Arial", 17),
            text_color=self.GRAY_TEXT
        )

        self.output_image_label.pack(
            fill="both",
            expand=True,
            padx=15,
            pady=10
        )

        # ====================================================
        # OPERATOR SECTION
        # ====================================================

        operator_section = ctk.CTkFrame(
            main_area,
            fg_color=self.WHITE,
            corner_radius=20,
            border_width=1,
            border_color=self.BORDER
        )

        operator_section.pack(
            fill="x",
            padx=25,
            pady=10
        )

        operator_title = ctk.CTkLabel(
            operator_section,
            text="🎨  SELECT IMAGE PROCESSING OPERATOR",
            font=("Arial", 17, "bold"),
            text_color=self.DARK_TEXT
        )

        operator_title.pack(
            pady=(15, 10)
        )

        # ====================================================
        # OPERATOR BUTTONS
        # ====================================================

        button_frame = ctk.CTkFrame(
            operator_section,
            fg_color="transparent"
        )

        button_frame.pack(
            pady=(0, 15)
        )

        # Mean
        mean_button = ctk.CTkButton(
            button_frame,
            text="🟨  MEAN FILTER",
            width=175,
            height=48,
            corner_radius=12,
            fg_color=self.YELLOW,
            hover_color=self.YELLOW_HOVER,
            text_color=self.DARK_TEXT,
            font=("Arial", 13, "bold"),
            command=lambda: self.select_operator("Mean")
        )

        mean_button.pack(
            side="left",
            padx=5
        )

        # Gaussian
        gaussian_button = ctk.CTkButton(
            button_frame,
            text="🟪  GAUSSIAN",
            width=175,
            height=48,
            corner_radius=12,
            fg_color=self.PURPLE,
            hover_color=self.PURPLE_HOVER,
            text_color=self.WHITE,
            font=("Arial", 13, "bold"),
            command=lambda: self.select_operator("Gaussian")
        )

        gaussian_button.pack(
            side="left",
            padx=5
        )

        # Sobel
        sobel_button = ctk.CTkButton(
            button_frame,
            text="🟧  SOBEL",
            width=175,
            height=48,
            corner_radius=12,
            fg_color=self.ORANGE,
            hover_color=self.ORANGE_HOVER,
            text_color=self.WHITE,
            font=("Arial", 13, "bold"),
            command=lambda: self.select_operator("Sobel")
        )

        sobel_button.pack(
            side="left",
            padx=5
        )

        # Laplacian
        laplacian_button = ctk.CTkButton(
            button_frame,
            text="🩷  LAPLACIAN",
            width=175,
            height=48,
            corner_radius=12,
            fg_color=self.PINK,
            hover_color=self.PINK_HOVER,
            text_color=self.WHITE,
            font=("Arial", 13, "bold"),
            command=lambda: self.select_operator("Laplacian")
        )

        laplacian_button.pack(
            side="left",
            padx=5
        )

        # Apply
        apply_button = ctk.CTkButton(
            button_frame,
            text="🟩  APPLY OPERATOR",
            width=190,
            height=48,
            corner_radius=12,
            fg_color=self.GREEN,
            hover_color=self.GREEN_HOVER,
            text_color=self.WHITE,
            font=("Arial", 13, "bold"),
            command=self.apply_operator
        )

        apply_button.pack(
            side="left",
            padx=12
        )

        # ====================================================
        # SELECTED OPERATOR LABEL
        # ====================================================

        self.selected_label = ctk.CTkLabel(
            operator_section,
            text="Selected Operator: MEAN FILTER",
            font=("Arial", 13, "bold"),
            text_color=self.YELLOW_HOVER
        )

        self.selected_label.pack(
            pady=(0, 12)
        )

        # ====================================================
        # INFORMATION AREA
        # ====================================================

        info_frame = ctk.CTkFrame(
            main_area,
            fg_color=self.LIGHT_BG,
            height=260
        )

        info_frame.pack(
            fill="x",
            padx=25,
            pady=(0, 20)
        )

        info_frame.pack_propagate(False)

        # ====================================================
        # KERNEL CARD
        # ====================================================

        kernel_card = ctk.CTkFrame(
            info_frame,
            fg_color=self.WHITE,
            corner_radius=20,
            border_width=3,
            border_color=self.PURPLE
        )

        kernel_card.pack(
            side="left",
            fill="both",
            expand=True,
            padx=(0, 8)
        )

        kernel_title = ctk.CTkLabel(
            kernel_card,
            text="🧮  KERNEL / MASK",
            font=("Arial", 17, "bold"),
            text_color=self.PURPLE
        )

        kernel_title.pack(
            pady=10
        )

        self.kernel_text = ctk.CTkTextbox(
            kernel_card,
            font=("Courier New", 13),
            fg_color=self.PURPLE_LIGHT,
            text_color=self.DARK_TEXT,
            corner_radius=12,
            border_width=0
        )

        self.kernel_text.pack(
            fill="both",
            expand=True,
            padx=12,
            pady=(0, 12)
        )

        self.kernel_text.insert(
            "1.0",
            "Select an operator\n\nto display its kernel."
        )

        # ====================================================
        # PIXEL MATRIX CARD
        # ====================================================

        pixel_card = ctk.CTkFrame(
            info_frame,
            fg_color=self.WHITE,
            corner_radius=20,
            border_width=3,
            border_color=self.CYAN
        )

        pixel_card.pack(
            side="right",
            fill="both",
            expand=True,
            padx=(8, 0)
        )

        pixel_title = ctk.CTkLabel(
            pixel_card,
            text="🔢  PIXEL MATRIX",
            font=("Arial", 17, "bold"),
            text_color=self.CYAN
        )

        pixel_title.pack(
            pady=10
        )

        self.pixel_text = ctk.CTkTextbox(
            pixel_card,
            font=("Courier New", 10),
            fg_color=self.CYAN_LIGHT,
            text_color=self.DARK_TEXT,
            corner_radius=12,
            border_width=0
        )

        self.pixel_text.pack(
            fill="both",
            expand=True,
            padx=12,
            pady=(0, 12)
        )

        self.pixel_text.insert(
            "1.0",
            "Upload an image\n\nto display pixel values."
        )

        # ====================================================
        # ADDITIONAL INFORMATION CARD
        # ====================================================

        details_card = ctk.CTkFrame(
            main_area,
            fg_color=self.WHITE,
            corner_radius=20,
            border_width=1,
            border_color=self.BORDER
        )

        details_card.pack(
            fill="x",
            padx=25,
            pady=(0, 20)
        )

        details_title = ctk.CTkLabel(
            details_card,
            text="📊  IMAGE PROCESSING INFORMATION",
            font=("Arial", 16, "bold"),
            text_color=self.MAGENTA
        )

        details_title.pack(
            pady=(12, 5)
        )

        self.details_label = ctk.CTkLabel(
            details_card,
            text=(
                "Upload an image to view processing information."
            ),
            font=("Arial", 12),
            text_color=self.GRAY_TEXT
        )

        self.details_label.pack(
            pady=(0, 15)
        )

        # ====================================================
        # STATUS BAR
        # ====================================================

        self.status = ctk.CTkLabel(
            self,
            text="●  Ready — Upload an image to begin",
            height=35,
            anchor="w",
            font=("Arial", 12, "bold"),
            text_color=self.GRAY_TEXT,
            fg_color=self.WHITE
        )

        self.status.pack(
            fill="x",
            side="bottom"
        )

    # ========================================================
    # SELECT OPERATOR
    # ========================================================

    def select_operator(self, operator):

        self.selected_operator = operator

        self.selected_label.configure(
            text=f"Selected Operator: {operator.upper()}"
        )

        self.status.configure(
            text=f"●  {operator} operator selected"
        )

        # Show kernel immediately
        self.show_kernel(operator)

    # ========================================================
    # SHOW KERNEL
    # ========================================================

    def show_kernel(self, operator):

        self.kernel_text.delete(
            "1.0",
            "end"
        )

        if operator == "Mean":

            text = (
                "MEAN FILTER — 3 × 3\n\n"
                "[ 0.111  0.111  0.111 ]\n"
                "[ 0.111  0.111  0.111 ]\n"
                "[ 0.111  0.111  0.111 ]"
            )

        elif operator == "Gaussian":

            kernel = cv2.getGaussianKernel(
                3,
                0
            )

            gaussian_kernel = (
                kernel @ kernel.T
            )

            text = (
                "GAUSSIAN FILTER — 3 × 3\n\n"
                + np.array2string(
                    gaussian_kernel,
                    precision=4,
                    suppress_small=True
                )
            )

        elif operator == "Sobel":

            text = (
                "SOBEL OPERATOR\n\n"
                "SOBEL X:\n"
                "[ -1   0   1 ]\n"
                "[ -2   0   2 ]\n"
                "[ -1   0   1 ]\n\n"
                "SOBEL Y:\n"
                "[ -1  -2  -1 ]\n"
                "[  0   0   0 ]\n"
                "[  1   2   1 ]"
            )

        elif operator == "Laplacian":

            text = (
                "LAPLACIAN OPERATOR — 3 × 3\n\n"
                "[  0   1   0 ]\n"
                "[  1  -4   1 ]\n"
                "[  0   1   0 ]"
            )

        else:

            text = "Select an operator."

        self.kernel_text.insert(
            "1.0",
            text
        )

    # ========================================================
    # UPLOAD IMAGE
    # ========================================================

    def upload_image(self):

        file_path = filedialog.askopenfilename(
            title="Select Image",
            filetypes=[
                (
                    "Image Files",
                    "*.jpg *.jpeg *.png *.bmp *.tif *.tiff"
                )
            ]
        )

        if not file_path:
            return

        self.image = cv2.imread(
            file_path
        )

        if self.image is None:

            messagebox.showerror(
                "Error",
                "Unable to read the selected image."
            )

            return

        # Convert to grayscale

        self.gray_image = cv2.cvtColor(
            self.image,
            cv2.COLOR_BGR2GRAY
        )

        # Image dimensions

        height, width = (
            self.gray_image.shape
        )

        # File name

        import os

        file_name = os.path.basename(
            file_path
        )

        self.image_info.configure(
            text=(
                f"📐 {file_name}   |   "
                f"{width} × {height} pixels"
            )
        )

        # Display input image

        self.display_image(
            self.image,
            self.input_image_label
        )

        # Display original pixels

        self.display_pixels(
            self.gray_image
        )

        # Information

        self.details_label.configure(
            text=(
                f"Image: {file_name}     •     "
                f"Width: {width}px     •     "
                f"Height: {height}px     •     "
                f"Channels: 1 (Grayscale)     •     "
                f"Pixel Range: 0–255"
            )
        )

        # Reset output

        self.output_image = None

        self.output_image_label.configure(
            image=None,
            text=(
                "⚙️\n\n"
                "Choose an operator\n"
                "and click APPLY"
            )
        )

        # Show currently selected kernel

        self.show_kernel(
            self.selected_operator
        )

        self.status.configure(
            text="●  Image uploaded successfully ✓"
        )

    # ========================================================
    # DISPLAY IMAGE
    # ========================================================

    def display_image(
        self,
        image,
        label
    ):

        # Convert BGR → RGB

        if len(image.shape) == 2:

            image_rgb = cv2.cvtColor(
                image,
                cv2.COLOR_GRAY2RGB
            )

        else:

            image_rgb = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            )

        pil_image = Image.fromarray(
            image_rgb
        )

        # Keep image inside card

        max_width = 600
        max_height = 310

        pil_image.thumbnail(
            (
                max_width,
                max_height
            )
        )

        ctk_image = ctk.CTkImage(
            light_image=pil_image,
            dark_image=pil_image,
            size=pil_image.size
        )

        label.configure(
            image=ctk_image,
            text=""
        )

        label.image = ctk_image

    # ========================================================
    # DISPLAY PIXELS
    # ========================================================

    def display_pixels(
        self,
        image
    ):

        self.pixel_text.delete(
            "1.0",
            "end"
        )

        height, width = image.shape

        self.pixel_text.insert(
            "end",
            f"Dimensions: {width} × {height}\n\n"
        )

        # Display first 15 × 20 pixels

        rows = min(
            height,
            15
        )

        cols = min(
            width,
            20
        )

        for i in range(rows):

            row = image[
                i,
                :cols
            ]

            values = " ".join(
                f"{int(v):3d}"
                for v in row
            )

            self.pixel_text.insert(
                "end",
                values + "\n"
            )

        if height > 15 or width > 20:

            self.pixel_text.insert(
                "end",
                "\n[First 15 × 20 pixels shown]"
            )

    # ========================================================
    # APPLY OPERATOR
    # ========================================================

    def apply_operator(self):

        if self.gray_image is None:

            messagebox.showwarning(
                "No Image",
                "Please upload an image first."
            )

            return

        operator = self.selected_operator

        # ====================================================
        # MEAN FILTER
        # ====================================================

        if operator == "Mean":

            kernel = (
                np.ones(
                    (3, 3),
                    dtype=np.float32
                ) / 9
            )

            self.output_image = cv2.filter2D(
                self.gray_image,
                -1,
                kernel
            )

            kernel_text = (
                "MEAN FILTER — 3 × 3\n\n"
                "[ 0.111  0.111  0.111 ]\n"
                "[ 0.111  0.111  0.111 ]\n"
                "[ 0.111  0.111  0.111 ]"
            )

        # ====================================================
        # GAUSSIAN FILTER
        # ====================================================

        elif operator == "Gaussian":

            gaussian_1d = cv2.getGaussianKernel(
                3,
                0
            )

            kernel = (
                gaussian_1d @ gaussian_1d.T
            )

            self.output_image = cv2.GaussianBlur(
                self.gray_image,
                (3, 3),
                0
            )

            kernel_text = (
                "GAUSSIAN FILTER — 3 × 3\n\n"
                + np.array2string(
                    kernel,
                    precision=4,
                    suppress_small=True
                )
            )

        # ====================================================
        # SOBEL
        # ====================================================

        elif operator == "Sobel":

            sobel_x = cv2.Sobel(
                self.gray_image,
                cv2.CV_64F,
                1,
                0,
                ksize=3
            )

            sobel_y = cv2.Sobel(
                self.gray_image,
                cv2.CV_64F,
                0,
                1,
                ksize=3
            )

            magnitude = cv2.magnitude(
                sobel_x.astype(np.float32),
                sobel_y.astype(np.float32)
            )

            self.output_image = cv2.convertScaleAbs(
                magnitude
            )

            kernel_text = (
                "SOBEL OPERATOR\n\n"
                "SOBEL X:\n"
                "[ -1   0   1 ]\n"
                "[ -2   0   2 ]\n"
                "[ -1   0   1 ]\n\n"
                "SOBEL Y:\n"
                "[ -1  -2  -1 ]\n"
                "[  0   0   0 ]\n"
                "[  1   2   1 ]"
            )

        # ====================================================
        # LAPLACIAN
        # ====================================================

        elif operator == "Laplacian":

            kernel = np.array(
                [
                    [0, 1, 0],
                    [1, -4, 1],
                    [0, 1, 0]
                ],
                dtype=np.float32
            )

            result = cv2.filter2D(
                self.gray_image,
                cv2.CV_64F,
                kernel
            )

            self.output_image = cv2.convertScaleAbs(
                result
            )

            kernel_text = (
                "LAPLACIAN OPERATOR — 3 × 3\n\n"
                "[  0   1   0 ]\n"
                "[  1  -4   1 ]\n"
                "[  0   1   0 ]"
            )

        else:

            return

        # ====================================================
        # DISPLAY KERNEL
        # ====================================================

        self.kernel_text.delete(
            "1.0",
            "end"
        )

        self.kernel_text.insert(
            "1.0",
            kernel_text
        )

        # ====================================================
        # DISPLAY OUTPUT
        # ====================================================

        self.display_image(
            self.output_image,
            self.output_image_label
        )

        # ====================================================
        # DISPLAY OUTPUT PIXELS
        # ====================================================

        self.display_pixels(
            self.output_image
        )

        # ====================================================
        # UPDATE INFORMATION
        # ====================================================

        height, width = (
            self.output_image.shape
        )

        self.details_label.configure(
            text=(
                f"Operator: {operator}     •     "
                f"Output Size: {width} × {height}     •     "
                f"Input: Grayscale     •     "
                f"Processing: 3 × 3 Kernel"
            )
        )

        self.status.configure(
            text=(
                f"●  {operator} operator applied "
                f"successfully ✓"
            )
        )

    # ========================================================
    # RESET
    # ========================================================

    def reset(self):

        self.image = None
        self.gray_image = None
        self.output_image = None

        self.selected_operator = "Mean"

        # Input

        self.input_image_label.configure(
            image=None,
            text=(
                "📷\n\n"
                "Upload an image to begin"
            )
        )

        # Output

        self.output_image_label.configure(
            image=None,
            text=(
                "⚙️\n\n"
                "Choose an operator\n"
                "and click APPLY"
            )
        )

        # Kernel

        self.kernel_text.delete(
            "1.0",
            "end"
        )

        self.kernel_text.insert(
            "1.0",
            "Select an operator\n\nto display its kernel."
        )

        # Pixels

        self.pixel_text.delete(
            "1.0",
            "end"
        )

        self.pixel_text.insert(
            "1.0",
            "Upload an image\n\nto display pixel values."
        )

        # Information

        self.image_info.configure(
            text="No image loaded"
        )

        self.details_label.configure(
            text=(
                "Upload an image to view "
                "processing information."
            )
        )

        self.selected_label.configure(
            text="Selected Operator: MEAN FILTER"
        )

        self.status.configure(
            text="●  Ready — Upload an image to begin"
        )


# ============================================================
# START APPLICATION
# ============================================================

if __name__ == "__main__":

    app = ImageProcessingApp()

    app.mainloop()